### Passo 1: Processamento de Dados com spaCy

#### Instalação de Ambiente:

In [19]:
%pip install pandas spacy scikit-learn matplotlib seaborn wordcloud

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [20]:
!python -m spacy download en_core_web_sm

Traceback (most recent call last):
  File "c:\Users\Julia\AppData\Local\Programs\Python\Python311\Lib\site-packages\urllib3\connection.py", line 198, in _new_conn
    sock = connection.create_connection(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Julia\AppData\Local\Programs\Python\Python311\Lib\site-packages\urllib3\util\connection.py", line 60, in create_connection
    for res in socket.getaddrinfo(host, port, family, socket.SOCK_STREAM):
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Julia\AppData\Local\Programs\Python\Python311\Lib\socket.py", line 962, in getaddrinfo
    for res in _socket.getaddrinfo(host, port, family, type, proto, flags):
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
socket.gaierror: [Errno 11001] getaddrinfo failed

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "c:\Users\Julia\AppData\Local\Programs\Python\Py

#### Importação de Libs

In [21]:
import pandas as pd
import spacy
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix
from wordcloud import WordCloud 

nlp = spacy.load("en_core_web_sm")

#### Criação e Consumo dos Datasets

In [22]:
# Utilização de Dataset Local (arquivo bruto)
df = pd.read_csv('../../dataset/raw/phishing_emails.csv', encoding='utf-8')  

# Visualização prévia dele
df.head()

,Unnamed: 0,Email Text,Email Type
0,0,"re : 6 . 1100 , disc : uniformitarianism , re ...",Safe Email
1,1,the other side of * galicismos * * galicismo *...,Safe Email
2,2,re : equistar deal tickets are you still avail...,Safe Email
3,3,\nHello I am your hot lil horny toy.\n I am...,Phishing Email
4,4,software at incredibly low prices ( 86 % lower...,Phishing Email


In [23]:
# Utilização de Dataset Local (arquivo bruto)
df_02 = pd.read_csv('../../dataset/raw/scam_1.csv', encoding='utf-8')  

# Visualização prévia dele
df_02.head()

,Unnamed: 0,0
0,0,Your social security number has been suspended...
1,1,"Hello, is this Bob Smith?"
2,2,Hi. I’m a representative of the Social Securit...
3,3,"Hello Mr/Mrs. X, my name is Y and I work with ..."
4,4,I'm speaking from the Department of Homeland S...


In [24]:
# Utilização de Dataset Local (arquivo bruto)
df_03 = pd.read_csv('../../dataset/raw/scam_2.csv', encoding='utf-8')  

# Visualização prévia dele
df_03.head()

,Unnamed: 0,0
0,0,There has been a lawsuit filed against you. Pl...
1,1,This is Bernson and Bailey. We are currently r...
2,2,This is an official notification that legal ac...
3,3,"Hello Mr/Mrs. X, my name is Y and I work in th..."
4,4,Your car had been parked in a no-parking zone ...


In [25]:
# Utilização de Dataset Local (arquivo bruto)
df_04 = pd.read_csv('../../dataset/raw/scam_3.csv', encoding='utf-8')  

# Visualização prévia dele
df_04.head()

,Unnamed: 0,0
0,0,You’re saying you never received any documents...
1,1,"Sir, you still owe $4,308 on your taxes. You s..."
2,2,The IRS documentation shows that you’ve receiv...
3,3,"Hello Mr/Mrs. X, my name is Y and I work in th..."
4,4,Am I speaking to Mr. John Doe?


In [26]:
# Utilização de Dataset Local (arquivo bruto)
df_05 = pd.read_csv('../../dataset/raw/scam_4.csv', encoding='utf-8')  

# Visualização prévia dele
df_05.head()

,Unnamed: 0,0
0,0,This message is urgent. Please listen closely....
1,1,"Hello, due to a mismatch on your employment re..."
2,2,Your bank accounts and tax returns are under r...
3,3,"Hello, my name is X, and I am a representative..."
4,4,Please listen to this message in its entirety....


In [27]:
# Utilização de Dataset Local (arquivo bruto)
df_06 = pd.read_csv('../../dataset/raw/scam_5.csv', encoding='utf-8')  

# Visualização prévia dele
df_06.head()

,Unnamed: 0,0
0,0,"Hello, is this Mrs. Johnson? My name is Tim Th..."
1,1,"Hi! My name is Josie, calling on behalf of the..."
2,2,"Hi, Mrs. Johnson. This is Tim Bono. Great news..."
3,3,"Hi, am I speaking to Mr/Mrs. X? My name is Y, ..."
4,4,Hey my name is Roger Doe and am I speaking Mr....


In [28]:
# Utilização de Dataset Local (arquivo bruto)
df_07 = pd.read_csv('../../dataset/raw/emotional_social_engineering_attacks.csv', encoding='utf-8')  

# Visualização prévia dele
df_07.head()

,Chat Log,Result
0,"Hey, I heard your company is downsizing. Are y...",Attack
1,Can you help me with something urgent? I reall...,Attack
2,Congratulations! You've won a prize! Just need...,Attack
3,I can't believe what happened to your colleagu...,Attack
4,Your account has been compromised. Please veri...,Attack


#### Pré-Processamento do .CSV "Bruto"

In [29]:
import pandas as pd
import spacy

# Carrega o modelo spaCy
nlp = spacy.load("en_core_web_sm")

# Função para limpar e processar o texto dos e-mails usando spaCy:
# - converte para minúsculas
# - remove stopwords, pontuações e tokens não alfabéticos
# - aplica lematização
def preprocess_spacy(text):
    if pd.isnull(text): 
        return ""
    doc = nlp(text.lower())
    tokens = [token.lemma_ for token in doc if not token.is_stop and not token.is_punct and token.is_alpha]
    return " ".join(tokens)

In [30]:
# -----------------------------
# Dataset principal (df)
# -----------------------------
df = pd.read_csv('../../dataset/raw/phishing_emails.csv', encoding='utf-8')

# Remove coluna extra se existir
if "Unnamed: 0" in df.columns:
    df.drop(columns=["Unnamed: 0"], inplace=True)

# Padroniza labels
df["Engenharia Social?"] = df["Email Type"].replace({
    "Safe Email": 0,
    "Phishing Email": 1
})

# Renomeia e mantém só as colunas necessárias
df.rename(columns={"Email Text": "Conteudo"}, inplace=True)
df = df[["Conteudo", "Engenharia Social?"]]

In [31]:
# -----------------------------
# Datasets df02 até df06 (só têm coluna "0")
# Todos são ataques → label 1
# -----------------------------
datasets_scam = []
for i in range(1, 6):  # scam_1.csv até scam_5.csv
    temp = pd.read_csv(f'../../dataset/raw/scam_{i}.csv', encoding='utf-8')
    if "Unnamed: 0" in temp.columns:
        temp.drop(columns=["Unnamed: 0"], inplace=True)
    temp.rename(columns={temp.columns[0]: "Conteudo"}, inplace=True)
    temp["Engenharia Social?"] = 1  # todos são ataques
    datasets_scam.append(temp)

In [32]:
# -----------------------------
# Dataset df07 (emotional attacks)
# Coluna "Chat Log" → Conteudo
# Coluna "Result" é sempre "Attack" → 1
# -----------------------------
df_07 = pd.read_csv('../../dataset/raw/emotional_social_engineering_attacks.csv', encoding='utf-8')

df_07.rename(columns={"Chat Log": "Conteudo"}, inplace=True)
df_07["Engenharia Social?"] = df_07["Result"].replace({
    "Attack": 1,
    "No Attack": 0
})
df_07 = df_07[["Conteudo", "Engenharia Social?"]]

In [33]:
# -----------------------------
# Junta todos os datasets
# -----------------------------
final_df = pd.concat([df] + datasets_scam + [df_07], ignore_index=True)

In [34]:
# -----------------------------
# Pré-processa os textos
# -----------------------------
final_df["Conteudo"] = final_df["Conteudo"].apply(preprocess_spacy)

In [35]:
# Salva no CSV final
final_df.to_csv("../../dataset/processed/emails_unificado.csv", index=False, encoding="utf-8")

In [36]:
print(final_df.head())
print(final_df["Engenharia Social?"].value_counts())

                                            Conteudo  Engenharia Social?
0  disc uniformitarianism sex lang dick hudson ob...                   0
1  galicismo galicismo spanish term name improper...                   0
2  equistar deal ticket available assist robert e...                   0
3  hello hot lil horny toy dream open minded pers...                   1
4  software incredibly low price low drapery seve...                   1
Engenharia Social?
0    1875
1    1507
Name: count, dtype: int64
